# Визуальный pipeline на 4 изображениях: сглаживание перед поиском аномалий

Этот notebook повторяет `four_image_visual_pipeline.ipynb`, но использует актуальный порядок обработки из основного `full_pipeline.py`:

1. исходные фотографии без подписей;
2. детектор чашки Петри;
3. crop/resize до 736x736;
4. YOLO26x-seg instance segmentation;
5. сглаживание бинарных масок колоний;
6. извлечение признаков и поиск аномалий уже по сглаженным маскам;
7. усиленное сравнение внутри морфотипа;
8. отдельный статус `review_segmentation` для сомнительных масок;
9. `stability check` при небольших изменениях яркости/контраста/blur;
10. визуализация общего highlight-набора всех статусов с лимитом 20%, но не более 20 объектов на изображение.

Старый notebook оставлен без изменений, чтобы можно было сравнить результаты до/после сглаживания и финальных фильтров.


## 1. Импорт библиотек

Используются готовые функции из `full_pipline/full_pipeline.py`, чтобы не дублировать основной код извлечения признаков и расчёта anomaly score.


In [ ]:
from pathlib import Path
import sys
import math
import colorsys
import json

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import display, Image as IPyImage
from skimage.measure import find_contours

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "full_pipline":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from full_pipline.full_pipeline import (
    make_full_pipeline_config,
    load_pipeline_models,
    run_single_image_pipeline as run_single_image_pipeline_current,
    select_visual_highlight_ids,
    read_image_rgb,
    prepare_image_for_segmentation,
    predict_colony_masks,
    extract_colony_features,
    compute_anomaly_scores,
    explain_anomaly,
    compute_plate_quality,
    select_top_anomalies,
    collect_review_candidates,
    build_feature_correlation_report,
    save_feature_space_pca,
    save_colony_crops,
    clean_instance_mask,
    normalize_masks_input,
    visualize_anomalies,
)

print("Корень проекта:", PROJECT_ROOT)


## 2. Настройки

`MAX_ANOMALY_VISUALIZATION_FRACTION` ограничивает общее число визуально выделенных объектов на изображении. Значение `0.20` означает не больше 20% от общего числа найденных колоний с учётом всех визуальных статусов: `select_candidate`, `review_segmentation`, `unstable_candidate` и других review-классов. Дополнительно действует абсолютный предел `MAX_ANOMALY_VISUALIZATION_COUNT = 20`. Это верхний предел, а не обязательное число: edge-aware и diversity-фильтры могут оставить меньше объектов.

Notebook использует тот же финальный фильтр, что и EXE: сглаживание масок до расчёта признаков, усиленный вес морфотипа, `review_segmentation` для сомнительных масок, `stability check` по яркости/контрасту/blur, spatial diversity, edge-aware cap для края чашки и отдельный cap для `review_segmentation`.


In [ ]:
# Четыре изображения, которые будут использоваться на всех этапах визуализации.
IMAGE_PATHS = [
    Path(r"C:/ColonyNet/Петри/IMG_4655.jpg"),
    Path(r"C:/ColonyNet/Петри/IMG_6137.jpg"),
    Path(r"C:/ColonyNet/Петри/IMG_4377.jpg"),
    Path(r"C:/ColonyNet/Петри/IMG_7438.jpg"),
]

# Папка для сохранения визуализаций и таблиц этого notebook.
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "colony_anomaly_detection" / "four_image_visual_pipeline_smoothed_anomaly"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Новый вариант алгоритма: перед извлечением признаков и anomaly scoring
# каждая YOLO instance mask сглаживается GaussianBlur и снова бинаризуется.
APPLY_MASK_SMOOTHING_BEFORE_ANOMALY = True
MASK_SMOOTH_SIGMA = 1.6
MASK_SMOOTH_THRESHOLD = 0.50

# Главное ограничение визуализации: не больше 20% найденных колоний на чашке
# с учётом всех визуальных статусов.
MAX_ANOMALY_VISUALIZATION_FRACTION = 0.20

# Дополнительный абсолютный лимит. None означает, что работает только процентный лимит.
MAX_ANOMALY_VISUALIZATION_COUNT = 20

# По умолчанию не показываем минимум 1 объект, если 20% даёт 0.
# Если поставить 1, на чашках с малым числом колоний лимит 20% может быть превышен.
MIN_ANOMALIES_TO_SHOW = 0

# Не даём одной плотной зоне или одному морфотипу занять всю визуализацию.
MAX_VISUAL_HIGHLIGHTS_PER_NEIGHBORHOOD = 2
MAX_VISUAL_HIGHLIGHTS_PER_MORPHOTYPE = 2
VISUAL_HIGHLIGHT_EDGE_BAND_DIAMETERS = 2.0
MAX_VISUAL_HIGHLIGHTS_EDGE_FRACTION = 0.25
VISUAL_HIGHLIGHT_EDGE_SCORE_PENALTY = 0.20
MAX_VISUAL_REVIEW_SEGMENTATION_FRACTION = 0.20
MAX_VISUAL_REVIEW_SEGMENTATION_COUNT = 4

# Если True, review_candidates дополнительно могут попадать в список selected_ids.
# Оранжевая пунктирная обводка review-кандидатов показывается через visualize_anomalies(show_review=True).
INCLUDE_REVIEW_CANDIDATES_IN_VISUALIZATION = False

# Показывать численное значение score прямо на выделенной или review-колонии.
# Для визуализации как в improved notebook оставляем True.
SHOW_ANOMALY_SCORE_NUMBERS = True

# Сохранять ли PNG-файлы с сетками 2x2.
SAVE_STAGE_VISUALIZATIONS = True

# Толщина белой линии между изображениями в сетке, в пикселях.
GRID_GAP_PX = 4

# Внешняя белая рамка вокруг всей сетки. 0 = без внешней рамки.
GRID_OUTER_BORDER_PX = 0

# RGB-цвет фона сетки. По умолчанию - белый.
GRID_BACKGROUND_RGB = (255, 255, 255)

# Размер каждой плитки в сетке: (высота, ширина).
# Единый размер убирает толстые белые поля у исходных фото разного формата.
GRID_TILE_SIZE = (736, 736)

# cover = заполнить всю плитку без внутренних полей; contain = сохранить кадр с полями; resize = растянуть.
GRID_FIT_MODE = "cover"

# Флаг запуска моделей. Если не нужно заново выполнять inference, поставьте False.
RUN_PIPELINE = True

config = make_full_pipeline_config(output_dir=OUTPUT_DIR)

# Для демонстрационного notebook оставляем сохранение CSV, но отключаем лишние тяжёлые картинки из базового pipeline.
config.save_csv = True
config.save_xlsx = True
config.save_visualizations = False
config.save_colony_crops = False
config.save_feature_space_plot = False
config.save_feature_correlation_report = True

# Синхронизация notebook с текущим алгоритмом EXE/full_pipeline.py.
config.smooth_masks_before_anomaly = bool(APPLY_MASK_SMOOTHING_BEFORE_ANOMALY)
config.mask_smoothing_sigma = float(MASK_SMOOTH_SIGMA)
config.mask_smoothing_threshold = float(MASK_SMOOTH_THRESHOLD)
config.visual_highlight_percent = float(MAX_ANOMALY_VISUALIZATION_FRACTION)
config.visual_highlight_max_count = MAX_ANOMALY_VISUALIZATION_COUNT
config.visual_highlight_min_count = MIN_ANOMALIES_TO_SHOW
config.visual_highlight_max_per_neighborhood = MAX_VISUAL_HIGHLIGHTS_PER_NEIGHBORHOOD
config.visual_highlight_max_per_morphotype = MAX_VISUAL_HIGHLIGHTS_PER_MORPHOTYPE
config.visual_highlight_edge_band_diameters = VISUAL_HIGHLIGHT_EDGE_BAND_DIAMETERS
config.visual_highlight_max_edge_fraction = MAX_VISUAL_HIGHLIGHTS_EDGE_FRACTION
config.visual_highlight_edge_score_penalty = VISUAL_HIGHLIGHT_EDGE_SCORE_PENALTY
config.visual_highlight_max_review_segmentation_fraction = MAX_VISUAL_REVIEW_SEGMENTATION_FRACTION
config.visual_highlight_max_review_segmentation_count = MAX_VISUAL_REVIEW_SEGMENTATION_COUNT
config.morphotype_weight = 0.08
config.use_perturbation_stability = True
config.perturbation_stability_variants = (
    "brightness_plus_5pct",
    "contrast_minus_5pct",
    "slight_blur",
)
config.min_perturbation_stability_score = 0.66
config.min_perturbation_stability_trials = 2
config.perturbation_stability_match_distance = 20.0

# Исходные пути весов, из которых были скопированы модели в full_pipline/models.
ORIGINAL_PETRI_DETECTOR_WEIGHTS = Path(r"C:/ColonyNet/runs/detect/runs/petri_curcle/yolo26s_petri_curcle/weights/best.pt")
ORIGINAL_COLONY_SEG_WEIGHTS = Path(r"C:/ColonyNet/runs/segment/runs/segment/runs/colony_seg_mlflow_736/yolo26x-seg_cropped736_offline_aug/weights/best.pt")

print("Output dir:", OUTPUT_DIR)
print("Сглаживание перед поиском аномалий:", APPLY_MASK_SMOOTHING_BEFORE_ANOMALY, "sigma=", MASK_SMOOTH_SIGMA, "threshold=", MASK_SMOOTH_THRESHOLD)
print("Лимит визуального выделения по всем статусам:", f"{config.visual_highlight_percent:.0%}")
print("Максимум из одной плотной зоны:", config.visual_highlight_max_per_neighborhood)
print("Модель детектора чашки:", config.petri_detector_weights_path)
print("Модель сегментации колоний:", config.model_weights_path)
print("Детектор существует:", Path(config.petri_detector_weights_path).exists())
print("Сегментатор существует:", Path(config.model_weights_path).exists())
if ORIGINAL_PETRI_DETECTOR_WEIGHTS.exists():
    print(
        "Совпадает размер с исходным detector best.pt:",
        Path(config.petri_detector_weights_path).stat().st_size == ORIGINAL_PETRI_DETECTOR_WEIGHTS.stat().st_size,
    )
if ORIGINAL_COLONY_SEG_WEIGHTS.exists():
    print(
        "Совпадает размер с исходным YOLO26x-seg best.pt:",
        Path(config.model_weights_path).stat().st_size == ORIGINAL_COLONY_SEG_WEIGHTS.stat().st_size,
    )

for image_path in IMAGE_PATHS:
    print(image_path, "exists=", image_path.exists())


## 3. Вспомогательные функции визуализации

Все функции ниже не подписывают отдельные подграфики. Порядок изображений всегда соответствует списку `IMAGE_PATHS`.


In [ ]:
def _to_uint8_rgb(image):
    """Привести входное изображение к RGB uint8 без изменения геометрии."""
    arr = np.asarray(image)
    if arr.ndim == 2:
        arr = np.stack([arr, arr, arr], axis=-1)
    if arr.ndim != 3:
        raise ValueError(f"Ожидалось изображение HxW или HxWxC, получено shape={arr.shape}")
    if arr.shape[2] == 4:
        arr = arr[:, :, :3]
    if arr.shape[2] != 3:
        raise ValueError(f"Ожидалось 3 RGB-канала, получено shape={arr.shape}")
    if arr.dtype != np.uint8:
        arr = arr.astype(np.float32)
        finite = arr[np.isfinite(arr)]
        max_value = float(finite.max()) if finite.size else 0.0
        if max_value <= 1.0:
            arr = arr * 255.0
        arr = np.nan_to_num(arr, nan=0.0, posinf=255.0, neginf=0.0)
        arr = np.clip(arr, 0, 255).astype(np.uint8)
    return arr


def _resize_to_grid_tile(image, tile_size=None, fit_mode=None, background_rgb=None):
    """Привести изображение к общему размеру плитки без толстых полей."""
    image = _to_uint8_rgb(image)
    if tile_size is None:
        return image

    tile_h, tile_w = [int(v) for v in tile_size]
    if tile_h <= 0 or tile_w <= 0:
        raise ValueError(f"Некорректный GRID_TILE_SIZE: {tile_size}")

    fit_mode = GRID_FIT_MODE if fit_mode is None else str(fit_mode)
    background_rgb = GRID_BACKGROUND_RGB if background_rgb is None else background_rgb
    h, w = image.shape[:2]

    if fit_mode == "resize":
        return cv2.resize(image, (tile_w, tile_h), interpolation=cv2.INTER_AREA if max(h, w) > max(tile_h, tile_w) else cv2.INTER_LINEAR)

    if fit_mode == "contain":
        scale = min(tile_w / w, tile_h / h)
        new_w = max(1, int(round(w * scale)))
        new_h = max(1, int(round(h * scale)))
        resized = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_AREA if scale < 1 else cv2.INTER_LINEAR)
        bg = np.array(background_rgb, dtype=np.uint8).reshape(1, 1, 3)
        tile = np.zeros((tile_h, tile_w, 3), dtype=np.uint8)
        tile[:] = bg
        y0 = (tile_h - new_h) // 2
        x0 = (tile_w - new_w) // 2
        tile[y0:y0 + new_h, x0:x0 + new_w] = resized
        return tile

    if fit_mode != "cover":
        raise ValueError(f"Неизвестный GRID_FIT_MODE: {fit_mode}")

    scale = max(tile_w / w, tile_h / h)
    new_w = max(tile_w, int(round(w * scale)))
    new_h = max(tile_h, int(round(h * scale)))
    resized = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_AREA if scale < 1 else cv2.INTER_LINEAR)
    y0 = max(0, (new_h - tile_h) // 2)
    x0 = max(0, (new_w - tile_w) // 2)
    return resized[y0:y0 + tile_h, x0:x0 + tile_w]


def make_image_grid(
    images,
    n_cols=2,
    gap_px=None,
    outer_border_px=None,
    background_rgb=None,
    tile_size=None,
    fit_mode=None,
):
    """Собрать сетку изображений с одинаковыми плитками и точным зазором в пикселях."""
    if not images:
        raise ValueError("Список изображений пуст")

    gap_px = int(GRID_GAP_PX if gap_px is None else gap_px)
    outer_border_px = int(GRID_OUTER_BORDER_PX if outer_border_px is None else outer_border_px)
    background_rgb = GRID_BACKGROUND_RGB if background_rgb is None else background_rgb
    tile_size = GRID_TILE_SIZE if tile_size is None else tile_size
    fit_mode = GRID_FIT_MODE if fit_mode is None else fit_mode

    gap_px = max(0, gap_px)
    outer_border_px = max(0, outer_border_px)
    n_cols = max(1, int(n_cols))

    prepared = [
        _resize_to_grid_tile(image, tile_size=tile_size, fit_mode=fit_mode, background_rgb=background_rgb)
        for image in images
    ]
    n_rows = int(math.ceil(len(prepared) / n_cols))
    cell_h = max(image.shape[0] for image in prepared)
    cell_w = max(image.shape[1] for image in prepared)

    grid_h = outer_border_px * 2 + n_rows * cell_h + max(0, n_rows - 1) * gap_px
    grid_w = outer_border_px * 2 + n_cols * cell_w + max(0, n_cols - 1) * gap_px

    bg = np.array(background_rgb, dtype=np.uint8).reshape(1, 1, 3)
    grid = np.zeros((grid_h, grid_w, 3), dtype=np.uint8)
    grid[:] = bg

    for idx, image in enumerate(prepared):
        row = idx // n_cols
        col = idx % n_cols
        y0 = outer_border_px + row * (cell_h + gap_px)
        x0 = outer_border_px + col * (cell_w + gap_px)
        grid[y0:y0 + image.shape[0], x0:x0 + image.shape[1]] = image

    return grid


def show_four_images(
    images,
    save_path=None,
    figsize=(12, 12),
    gap_px=None,
    outer_border_px=None,
    background_rgb=None,
    tile_size=None,
    fit_mode=None,
):
    """Показать 4 изображения сеткой 2x2 с единой тонкой белой линией."""
    grid = make_image_grid(
        images,
        n_cols=2,
        gap_px=gap_px,
        outer_border_px=outer_border_px,
        background_rgb=background_rgb,
        tile_size=tile_size,
        fit_mode=fit_mode,
    )

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        ok = cv2.imwrite(str(save_path), cv2.cvtColor(grid, cv2.COLOR_RGB2BGR))
        if not ok:
            raise IOError(f"Не удалось сохранить изображение: {save_path}")
        display(IPyImage(filename=str(save_path)))
    else:
        fig, ax = plt.subplots(figsize=figsize)
        ax.imshow(grid)
        ax.set_axis_off()
        fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
        display(fig)
        plt.close(fig)

    return grid

def figure_to_rgb_image(fig, crop_to_axes=True):
    """Преобразовать Matplotlib Figure в RGB и убрать внешние поля оси."""
    fig.canvas.draw()
    rgba = np.asarray(fig.canvas.buffer_rgba())
    if crop_to_axes and fig.axes:
        bbox = fig.axes[0].get_window_extent()
        height, width = rgba.shape[:2]
        x0 = max(0, int(math.floor(bbox.x0)))
        x1 = min(width, int(math.ceil(bbox.x1)))
        y0 = max(0, int(math.floor(height - bbox.y1)))
        y1 = min(height, int(math.ceil(height - bbox.y0)))
        if x1 > x0 and y1 > y0:
            rgba = rgba[y0:y1, x0:x1]
    rgb = rgba[..., :3].copy()
    plt.close(fig)
    return rgb
def render_anomalies_like_original_notebook(image_rgb, masks, scores_df, selected_ids, highlight_ids=None, image_name=None):
    """Отрисовать аномалии тем же стилем, что и в `colony_anomaly_detection_yolo_x_improved.ipynb`.

    Стиль старого notebook:
    - все колонии имеют полупрозрачную заливку;
    - контуры masks сглаживаются через Gaussian blur;
    - выбранные аномалии выделяются жёлтым контуром;
    - под жёлтым контуром рисуется толстая чёрная обводка;
    - подписи не показываются, если `SHOW_ANOMALY_SCORE_NUMBERS=False`."""
    fig = visualize_anomalies(
        image_rgb=image_rgb,
        masks=masks,
        results_df=scores_df,
        selected_ids=selected_ids,
        save_path=None,
        title=None,
        show_scores=SHOW_ANOMALY_SCORE_NUMBERS,
        show_review=True,
        show_technical_warnings=False,
        highlight_ids=highlight_ids,
    )
    return figure_to_rgb_image(fig)


def draw_petri_bbox(image_rgb, preprocess_info, color=(255, 0, 0)):
    """Нарисовать bbox чашки Петри без текстовой подписи."""
    view = image_rgb.copy()
    xyxy = preprocess_info.get("crop_xyxy", None) if isinstance(preprocess_info, dict) else None
    detected = bool(preprocess_info.get("petri_detected", False)) if isinstance(preprocess_info, dict) else False
    if detected and xyxy is not None and len(xyxy) == 4:
        x1, y1, x2, y2 = [int(v) for v in xyxy]
        thickness = max(3, int(round(min(view.shape[:2]) * 0.006)))
        cv2.rectangle(view, (x1, y1), (x2, y2), color, thickness=thickness)
    return view


def _mask_color(index):
    hue = (0.13 + 0.61803398875 * index) % 1.0
    rgb = colorsys.hsv_to_rgb(hue, 0.78, 1.0)
    return np.array(rgb, dtype=float)


def overlay_instance_masks(image_rgb, masks, alpha=0.34):
    """Полупрозрачно залить masks и нарисовать сглаженные контуры без номеров.

    Сглаживание сделано так же, как в визуализации аномалий:
    бинарная маска переводится в float, размывается GaussianBlur,
    затем контур берётся по уровню `contour_level`. Это убирает ступеньки
    и делает края визуально мягче."""
    masks_list = normalize_masks_input(masks)
    base = image_rgb.astype(np.float32) / 255.0
    out = base.copy()
    h, w = image_rgb.shape[:2]
    smooth_sigma = 1.6
    contour_level = 0.35

    for idx, mask in enumerate(masks_list):
        mask_bool = np.asarray(mask).astype(bool)
        if mask_bool.shape != (h, w):
            mask_bool = cv2.resize(mask_bool.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST).astype(bool)
        color = _mask_color(idx)
        out[mask_bool] = (1.0 - alpha) * out[mask_bool] + alpha * color

    out_u8 = np.clip(out * 255, 0, 255).astype(np.uint8)

    fig, ax = plt.subplots(figsize=(8, 8), frameon=False)
    fig.patch.set_facecolor("black")
    fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
    ax.set_position([0, 0, 1, 1])
    ax.set_facecolor("black")
    ax.imshow(out_u8)
    ax.axis("off")

    for idx, mask in enumerate(masks_list):
        mask_bool = np.asarray(mask).astype(bool)
        if mask_bool.shape != (h, w):
            mask_bool = cv2.resize(mask_bool.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST).astype(bool)
        color = _mask_color(idx)
        mask_prob = mask_bool.astype(np.float32)
        mask_prob = cv2.GaussianBlur(mask_prob, (0, 0), sigmaX=smooth_sigma, sigmaY=smooth_sigma)
        for contour in find_contours(mask_prob, level=contour_level):
            ax.plot(
                contour[:, 1],
                contour[:, 0],
                color=color,
                linewidth=0.75,
                antialiased=True,
                solid_joinstyle="round",
                solid_capstyle="round",
            )

    return figure_to_rgb_image(fig)


def anomaly_visualization_limit(n_colonies):
    """Посчитать максимальное число выделяемых аномалий по процентному и абсолютному лимиту."""
    if n_colonies <= 0:
        return 0
    by_fraction = int(math.floor(n_colonies * float(MAX_ANOMALY_VISUALIZATION_FRACTION)))
    limit = max(int(MIN_ANOMALIES_TO_SHOW), by_fraction)
    if MAX_ANOMALY_VISUALIZATION_COUNT is not None:
        limit = min(limit, int(MAX_ANOMALY_VISUALIZATION_COUNT))
    return max(0, min(limit, int(n_colonies)))


def choose_anomaly_ids_for_visualization(scores_df, n_colonies):
    """Выбрать общий highlight-набор: максимум 20, edge-aware, без плотных подряд идущих групп."""
    return select_visual_highlight_ids(
        scores_df,
        n_colonies=n_colonies,
        max_fraction=MAX_ANOMALY_VISUALIZATION_FRACTION,
        max_count=MAX_ANOMALY_VISUALIZATION_COUNT,
        min_count=MIN_ANOMALIES_TO_SHOW,
        max_per_neighborhood=MAX_VISUAL_HIGHLIGHTS_PER_NEIGHBORHOOD,
        max_per_morphotype=MAX_VISUAL_HIGHLIGHTS_PER_MORPHOTYPE,
        edge_band_diameters=VISUAL_HIGHLIGHT_EDGE_BAND_DIAMETERS,
        max_edge_fraction=MAX_VISUAL_HIGHLIGHTS_EDGE_FRACTION,
        edge_score_penalty=VISUAL_HIGHLIGHT_EDGE_SCORE_PENALTY,
        max_review_segmentation_fraction=MAX_VISUAL_REVIEW_SEGMENTATION_FRACTION,
        max_review_segmentation_count=MAX_VISUAL_REVIEW_SEGMENTATION_COUNT,
    )


def overlay_limited_anomalies(image_rgb, masks, scores_df, selected_ids, show_score_numbers=False):
    """Нарисовать все колонии тонко, а выбранные аномалии выделить заметным контуром."""
    masks_list = normalize_masks_input(masks)
    out = image_rgb.copy()
    h, w = image_rgb.shape[:2]
    selected_set = {int(x) for x in selected_ids}

    # Сначала тонкие серые контуры всех найденных колоний.
    for idx, mask in enumerate(masks_list, start=1):
        mask_bool = np.asarray(mask).astype(bool)
        if mask_bool.shape != (h, w):
            mask_bool = cv2.resize(mask_bool.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST).astype(bool)
        contours, _ = cv2.findContours(mask_bool.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(out, contours, -1, (170, 170, 170), thickness=1, lineType=cv2.LINE_AA)

    # Затем выбранные аномалии: лёгкая заливка + толстый контур.
    score_lookup = {}
    if scores_df is not None and not scores_df.empty and "colony_id" in scores_df.columns:
        score_col = "final_anomaly_score_raw" if "final_anomaly_score_raw" in scores_df.columns else "final_anomaly_score"
        if score_col in scores_df.columns:
            score_lookup = {
                int(row["colony_id"]): float(row[score_col])
                for _, row in scores_df.dropna(subset=["colony_id"]).iterrows()
                if pd.notna(row.get(score_col, np.nan))
            }

    for idx, mask in enumerate(masks_list, start=1):
        if idx not in selected_set:
            continue
        mask_bool = np.asarray(mask).astype(bool)
        if mask_bool.shape != (h, w):
            mask_bool = cv2.resize(mask_bool.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST).astype(bool)

        yellow = np.array([255, 230, 0], dtype=np.uint8)
        out[mask_bool] = (0.78 * out[mask_bool].astype(np.float32) + 0.22 * yellow).astype(np.uint8)

        contours, _ = cv2.findContours(mask_bool.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(out, contours, -1, (0, 0, 0), thickness=5, lineType=cv2.LINE_AA)
        cv2.drawContours(out, contours, -1, (255, 230, 0), thickness=3, lineType=cv2.LINE_AA)

        if show_score_numbers:
            ys, xs = np.where(mask_bool)
            if len(xs) > 0:
                score = score_lookup.get(idx, np.nan)
                if np.isfinite(score):
                    cv2.putText(
                        out,
                        f"{score:.2f}",
                        (int(np.mean(xs)), int(np.mean(ys))),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.45,
                        (0, 0, 0),
                        2,
                        cv2.LINE_AA,
                    )
                    cv2.putText(
                        out,
                        f"{score:.2f}",
                        (int(np.mean(xs)), int(np.mean(ys))),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.45,
                        (255, 230, 0),
                        1,
                        cv2.LINE_AA,
                    )
    return out


## 3.1. Сглаживание и актуальный алгоритм аномалий

В исходном pipeline сглаживание применялось только при рисовании контуров. В этом варианте маски сглаживаются раньше: после YOLO-seg и базовой очистки, но до `extract_colony_features` и `compute_anomaly_scores`.

Дальше notebook вызывает текущий `run_single_image_pipeline` из `full_pipeline.py`, поэтому в демонстрации также работают новые финальные фильтры: усиленное сравнение внутри морфотипа, `review_segmentation` и `stability check`.


In [ ]:
def run_single_image_pipeline_smoothed_before_anomaly(
    image_path,
    model,
    config,
    petri_detector_model=None,
    smooth_sigma=1.6,
    smooth_threshold=0.50,
):
    """Notebook-wrapper вокруг актуального run_single_image_pipeline.

    Раньше этот notebook вручную дублировал часть pipeline, поэтому новые фильтры
    могли не попасть в демонстрацию. Теперь функция только фиксирует параметры
    notebook и вызывает основной код из full_pipeline.py.
    """
    config.smooth_masks_before_anomaly = bool(APPLY_MASK_SMOOTHING_BEFORE_ANOMALY)
    config.mask_smoothing_sigma = float(smooth_sigma)
    config.mask_smoothing_threshold = float(smooth_threshold)
    config.visual_highlight_percent = float(MAX_ANOMALY_VISUALIZATION_FRACTION)
    config.visual_highlight_max_count = MAX_ANOMALY_VISUALIZATION_COUNT
    config.visual_highlight_min_count = MIN_ANOMALIES_TO_SHOW
    config.visual_highlight_max_per_neighborhood = MAX_VISUAL_HIGHLIGHTS_PER_NEIGHBORHOOD
    config.visual_highlight_max_per_morphotype = MAX_VISUAL_HIGHLIGHTS_PER_MORPHOTYPE
    config.visual_highlight_edge_band_diameters = VISUAL_HIGHLIGHT_EDGE_BAND_DIAMETERS
    config.visual_highlight_max_edge_fraction = MAX_VISUAL_HIGHLIGHTS_EDGE_FRACTION
    config.visual_highlight_edge_score_penalty = VISUAL_HIGHLIGHT_EDGE_SCORE_PENALTY
    config.visual_highlight_max_review_segmentation_fraction = MAX_VISUAL_REVIEW_SEGMENTATION_FRACTION
    config.visual_highlight_max_review_segmentation_count = MAX_VISUAL_REVIEW_SEGMENTATION_COUNT
    config.morphotype_weight = 0.08
    config.use_perturbation_stability = True
    config.perturbation_stability_variants = (
        "brightness_plus_5pct",
        "contrast_minus_5pct",
        "slight_blur",
    )
    config.min_perturbation_stability_score = 0.66
    config.min_perturbation_stability_trials = 2
    config.perturbation_stability_match_distance = 20.0

    result = run_single_image_pipeline_current(
        image_path=image_path,
        model=model,
        config=config,
        petri_detector_model=petri_detector_model,
    )
    result["notebook_algorithm_notes"] = {
        "mask_smoothing_before_anomaly": bool(config.smooth_masks_before_anomaly),
        "mask_smoothing_sigma": float(config.mask_smoothing_sigma),
        "mask_smoothing_threshold": float(config.mask_smoothing_threshold),
        "visual_highlight_percent": float(config.visual_highlight_percent),
        "visual_highlight_max_per_neighborhood": int(config.visual_highlight_max_per_neighborhood),
        "visual_highlight_max_per_morphotype": int(config.visual_highlight_max_per_morphotype),
        "visual_highlight_edge_band_diameters": float(config.visual_highlight_edge_band_diameters),
        "visual_highlight_max_edge_fraction": float(config.visual_highlight_max_edge_fraction),
        "visual_highlight_max_review_segmentation_count": int(config.visual_highlight_max_review_segmentation_count),
        "morphotype_weight": float(config.morphotype_weight),
        "use_perturbation_stability": bool(config.use_perturbation_stability),
        "min_perturbation_stability_score": float(config.min_perturbation_stability_score),
        "min_perturbation_stability_trials": int(config.min_perturbation_stability_trials),
    }
    return result


## 4. Визуализация исходных изображений

Здесь показываются четыре исходных фотографии без подписей, осей и номеров.


In [ ]:
raw_images = []
for image_path in IMAGE_PATHS:
    raw_images.append(read_image_rgb(image_path))

fig_raw = show_four_images(
    raw_images,
    save_path=OUTPUT_DIR / "01_raw_images.png" if SAVE_STAGE_VISUALIZATIONS else None,
)


## 5. Полный запуск моделей

Эта ячейка последовательно запускает две модели и актуальный алгоритм анализа:

1. YOLO26s detector чашки Петри;
2. YOLO26x-seg instance segmentation колоний;
3. сглаживание масок перед расчётом признаков;
4. извлечение признаков, морфотипы и поиск аномалий;
5. `stability check` для первичных `select_candidate`.

Чтобы выполнить инференс, установите `RUN_PIPELINE = True` в ячейке настроек.


In [ ]:
results = {}

if RUN_PIPELINE:
    segmentation_model, petri_detector_model = load_pipeline_models(config)

    for image_path in tqdm(IMAGE_PATHS, desc="Полный pipeline со сглаживанием масок"):
        result = run_single_image_pipeline_smoothed_before_anomaly(
            image_path=image_path,
            model=segmentation_model,
            config=config,
            petri_detector_model=petri_detector_model,
            smooth_sigma=MASK_SMOOTH_SIGMA,
            smooth_threshold=MASK_SMOOTH_THRESHOLD,
        )
        results[image_path.name] = result

        # Базовая функция создаёт отдельную figure; в этом notebook мы строим свои сетки 2x2.
        if result.get("figure") is not None:
            plt.close(result["figure"])

    print("Pipeline со сглаживанием перед anomaly scoring выполнен для", len(results), "изображений")
else:
    print("RUN_PIPELINE = False. Чтобы запустить модели, установите RUN_PIPELINE = True в ячейке настроек.")


## 6. Визуализация применения детектора чашек

На каждом исходном изображении рисуется только bbox найденной чашки. Текстовые подписи не добавляются.


In [ ]:
if results:
    detector_views = []
    for image_path in IMAGE_PATHS:
        result = results[image_path.name]
        detector_views.append(draw_petri_bbox(result["image_original"], result["preprocess"]))

    fig_detector = show_four_images(
        detector_views,
        save_path=OUTPUT_DIR / "02_petri_detector_bbox.png" if SAVE_STAGE_VISUALIZATIONS else None,
    )
else:
    print("Сначала выполните pipeline: RUN_PIPELINE = True")


## 7. Визуализация crop 736x736 после детектора

Эти изображения уже передаются во вторую модель - YOLO26x-seg для instance segmentation колоний.


In [ ]:
if results:
    prepared_images = [results[p.name]["image"] for p in IMAGE_PATHS]
    fig_crops = show_four_images(
        prepared_images,
        save_path=OUTPUT_DIR / "03_petri_crops_736.png" if SAVE_STAGE_VISUALIZATIONS else None,
    )
else:
    print("Сначала выполните pipeline: RUN_PIPELINE = True")


## 8. Визуализация instance segmentation колоний

Все найденные колонии показаны полупрозрачной заливкой и тонкими контурами. Номера колоний не подписываются.


In [ ]:
if results:
    segmentation_views = []
    segmentation_summary = []

    for image_path in IMAGE_PATHS:
        result = results[image_path.name]
        masks = result["masks"]
        segmentation_views.append(overlay_instance_masks(result["image"], masks))
        segmentation_summary.append({
            "image_name": image_path.name,
            "n_detected_colonies": len(masks),
            "n_yolo_detections": len(result["detections"]),
        })

    fig_segmentation = show_four_images(
        segmentation_views,
        save_path=OUTPUT_DIR / "04_colony_instance_segmentation.png" if SAVE_STAGE_VISUALIZATIONS else None,
    )
    display(pd.DataFrame(segmentation_summary))
else:
    print("Сначала выполните pipeline: RUN_PIPELINE = True")


## 9. Поиск и визуализация аномальных колоний

На этом этапе используются `final_anomaly_score_raw`, `recommendation_status`, технические флаги, `perturbation_stability_pass` и правило ограничения визуализации.

Количество выделенных автоматических аномалий на чашке не превышает:

```text
floor(n_detected_colonies * MAX_ANOMALY_VISUALIZATION_FRACTION)
```

При значении `0.20` это не больше 20% от общего числа найденных колоний. Абсолютный предел `MAX_ANOMALY_VISUALIZATION_COUNT = 20` не даёт выделить больше 20 объектов на изображение. Визуализируются объекты разных статусов, но общий лимит считается суммарно, selector не берёт слишком много соседних колоний из одной плотной зоны, ограничивает долю объектов у края чашки и не даёт `review_segmentation` забить весь highlight-набор.


In [ ]:
if results:
    anomaly_views = []
    anomaly_summary = []

    for image_path in IMAGE_PATHS:
        result = results[image_path.name]
        n_colonies = len(result["masks"])
        limit = anomaly_visualization_limit(n_colonies)
        highlight_ids = choose_anomaly_ids_for_visualization(result["scores"], n_colonies)
        selected_df = result.get("selected", pd.DataFrame())
        selected_ids = (
            selected_df["colony_id"].dropna().astype(int).tolist()
            if isinstance(selected_df, pd.DataFrame) and not selected_df.empty and "colony_id" in selected_df.columns
            else []
        )

        anomaly_views.append(
            render_anomalies_like_original_notebook(
                image_rgb=result["image"],
                masks=result["masks"],
                scores_df=result["scores"],
                selected_ids=selected_ids,
                highlight_ids=highlight_ids,
                image_name=image_path.name,
            )
        )

        scores_df = result["scores"]
        n_valid = int(scores_df["valid_for_anomaly"].fillna(False).sum()) if not scores_df.empty and "valid_for_anomaly" in scores_df.columns else 0
        n_technical = int(scores_df["technical_warning"].fillna(False).sum()) if not scores_df.empty and "technical_warning" in scores_df.columns else 0
        status_counts = (
            scores_df["recommendation_status"].value_counts().to_dict()
            if not scores_df.empty and "recommendation_status" in scores_df.columns
            else {}
        )
        anomaly_summary.append({
            "image_name": image_path.name,
            "n_detected_colonies": n_colonies,
            "n_valid_colonies": n_valid,
            "n_selected_candidates": int(status_counts.get("select_candidate", 0)),
            "n_review_segmentation": int(status_counts.get("review_segmentation", 0)),
            "n_unstable_candidates": int(status_counts.get("unstable_candidate", 0)),
            "n_technical_warnings": n_technical,
            "max_allowed_visualized_anomalies": limit,
            "n_visualized_anomalies": len(highlight_ids),
            "visualized_colony_ids": highlight_ids,
        })

    fig_anomalies = show_four_images(
        anomaly_views,
        save_path=OUTPUT_DIR / "05_limited_anomaly_visualization.png" if SAVE_STAGE_VISUALIZATIONS else None,
    )

    anomaly_summary_df = pd.DataFrame(anomaly_summary)
    display(anomaly_summary_df)
    anomaly_summary_df.to_csv(OUTPUT_DIR / "limited_anomaly_visualization_summary.csv", index=False, encoding="utf-8-sig")
else:
    print("Сначала выполните pipeline: RUN_PIPELINE = True")


## 10. Таблицы результатов

Эта ячейка собирает краткие таблицы по selected/review/technical/stability объектам для всех четырёх изображений.


In [ ]:
if results:
    selected_tables = []
    visual_tables = []
    review_tables = []
    technical_tables = []
    stability_tables = []

    for image_path in IMAGE_PATHS:
        result = results[image_path.name]
        for key, target in [
            ("selected", selected_tables),
            ("visual_highlights", visual_tables),
            ("review_candidates", review_tables),
            ("technical_warnings", technical_tables),
            ("perturbation_stability", stability_tables),
        ]:
            df = result.get(key, pd.DataFrame())
            if df is not None and not df.empty:
                tmp = df.copy()
                tmp.insert(0, "image_name", image_path.name)
                target.append(tmp)

    all_selected = pd.concat(selected_tables, ignore_index=True) if selected_tables else pd.DataFrame()
    all_visual = pd.concat(visual_tables, ignore_index=True) if visual_tables else pd.DataFrame()
    all_review = pd.concat(review_tables, ignore_index=True) if review_tables else pd.DataFrame()
    all_technical = pd.concat(technical_tables, ignore_index=True) if technical_tables else pd.DataFrame()
    all_stability = pd.concat(stability_tables, ignore_index=True) if stability_tables else pd.DataFrame()

    all_selected.to_csv(OUTPUT_DIR / "all_selected_anomalies.csv", index=False, encoding="utf-8-sig")
    all_visual.to_csv(OUTPUT_DIR / "all_visual_highlighted_objects.csv", index=False, encoding="utf-8-sig")
    all_review.to_csv(OUTPUT_DIR / "all_review_candidates.csv", index=False, encoding="utf-8-sig")
    all_technical.to_csv(OUTPUT_DIR / "all_technical_warnings.csv", index=False, encoding="utf-8-sig")
    all_stability.to_csv(OUTPUT_DIR / "all_perturbation_stability.csv", index=False, encoding="utf-8-sig")

    print("Selected anomalies:", len(all_selected))
    display(all_selected.head(20))

    print("Visual highlights:", len(all_visual))
    display(all_visual.head(20))

    print("Review candidates:", len(all_review))
    display(all_review.head(20))

    print("Technical warnings:", len(all_technical))
    display(all_technical.head(20))

    print("Perturbation stability checks:", len(all_stability))
    display(all_stability.head(20))
else:
    print("Сначала выполните pipeline: RUN_PIPELINE = True")


## 11. Что сохраняется

После запуска notebook сохраняет:

- `01_raw_images.png` - четыре исходных изображения без подписей;
- `02_petri_detector_bbox.png` - применение детектора чашки;
- `03_petri_crops_736.png` - crop 736x736 после детектора;
- `04_colony_instance_segmentation.png` - instance masks колоний;
- `05_limited_anomaly_visualization.png` - общий highlight-набор всех статусов с ограничением 20% от числа колоний, но не более 20 объектов;
- `limited_anomaly_visualization_summary.csv` - сколько аномалий разрешено и показано по каждой чашке;
- `all_selected_anomalies.csv`;
- `all_visual_highlighted_objects.csv`;
- `all_review_candidates.csv`, включая `review_segmentation` и `unstable_candidate`;
- `all_technical_warnings.csv`;
- `all_perturbation_stability.csv`;
- внутри папки каждого изображения: `visual_highlighted_objects.csv`, `perturbation_stability.csv`, `mask_smoothing_diagnostics.csv`, `colony_anomaly_scores.csv`, `selected_anomalies.csv`, `review_candidates.csv`, `technical_warnings.csv`.

Папка сохранения задаётся переменной `OUTPUT_DIR`.
